# CMPE 272 In-Class: ERP Lab (BlueFin eFoils)

**Session 6, ERP, SCM and CRM. Section 49, Wednesday, September 23, 2026.**
**Group assignment: one notebook per project team (2 to 4 people). Due tonight, 11:59 PM. 1 point.**

BlueFin eFoils, Inc. is a fictional company. It builds and sells electric hydrofoil surfboards. A viral video
has pushed up demand for its BF-100 board kit, and stock is tight. In this notebook your team works through
the ERP processes from tonight's lecture on BlueFin's data: you map the Order-to-Cash and Procure-to-Pay
flows, allocate stock and create invoices, clean the vendor master, and check roles for segregation of duties.

## How to work on it

1. Go to <https://colab.research.google.com>, choose **File > Upload notebook**, and pick this file.
2. Choose **File > Save a copy in Drive before you type anything.** Edits to the uploaded file are lost when the tab closes.
3. Share your Drive copy with your teammates, so the whole team works in one notebook.
4. Run the setup cell. Then fill in every `REPLACE`, part by part.
5. After each code part, run its **CHECK** cell. It prints `PASS`, or `NOT YET` with what to fix.
6. Run the **validation cell** at the end. It lists anything that is still missing.
7. Choose **Runtime > Restart and run all**, and confirm that every cell has output.
8. Choose **File > Download > Download .ipynb**. One team member uploads that file to the Canvas assignment **In-Class: ERP Lab**.

## Parts and time

| Part | What you do | Time |
|---|---|---|
| 1 | Map Order-to-Cash and Procure-to-Pay in short written answers | 12 min |
| 2 | Order-to-Cash: allocate stock and create invoices | 12 min |
| 3 | Clean and de-duplicate the vendor master | 12 min |
| 4 | Roles and segregation of duties | 12 min |
| Optional | Procure-to-Pay reorders, a one-level MRP, a KPI plot (not graded) | if time |

Parts 1 to 4 are required. The optional sections are not graded.

**If something breaks, say so.** Start the answer field for that part with `BROKE:` and paste the error
message. That is an honest answer and it earns credit. A result you did not get is not.

## Your team

Write every member's name as it appears in Canvas, and your project team's name.

In [2]:
# Every member of your team, with each name exactly as it appears in Canvas (2 to 4 people).
GROUP = [
    "AMEYA MATHEW",
    "Olivia Hunter",
    "Salih Eren Yüzbaşıoğlu",
    # Add or delete lines so the list matches your team.
]
TEAM_NAME = "HIKE THE MODEL"

print("Team:", TEAM_NAME, "|", len(GROUP), "members")

Team: HIKE THE MODEL | 3 members


## 0) Setup: run this cell first

This cell builds the BlueFin dataset. Do not edit it. Running it again resets every table to the original data.
The date is fixed at September 23, 2026, so every team gets the same numbers.

In [3]:
# @title Setup: imports and the BlueFin dataset (run me, do not edit)
import pandas as pd
import numpy as np
import datetime as dt
import math
import textwrap
from collections import defaultdict

# Optional plotting, used only in the optional KPI section. Colab includes matplotlib.
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Random seed for reproducibility across teams
rng = np.random.default_rng(272)

# -------------------------------
# Scenario: BlueFin eFoils, Inc.
# -------------------------------
# BlueFin manufactures and sells electric hydrofoil surfboards (eFoils).
# Products:
#  - BF-100 Board Kit (finished good)
#  - BAT-9 Battery Pack
#  - CTRL-2 Control Unit
#  - FOIL-3 Hydrofoil Assembly
#  - CHR-1 Charger
# The BF-100 is assembled from BAT-9, CTRL-2, FOIL-3, and CHR-1.
# Sales spike is expected next month due to a viral video. Inventory is tight.

# A fixed date, so every team's numbers match the answer key.
today = pd.Timestamp("2026-09-23").normalize()

products = pd.DataFrame([
    {"sku":"BF-100","name":"BlueFin eFoil Board Kit","uom":"ea","unitCost":1800.00,"unitPrice":2999.00,"leadTimeDays":7,"reorderPoint":12,"reorderQty":20},
    {"sku":"BAT-9","name":"Battery Pack 9kWh","uom":"ea","unitCost":750.00,"unitPrice":999.00,"leadTimeDays":10,"reorderPoint":25,"reorderQty":40},
    {"sku":"CTRL-2","name":"Control Unit Mk2","uom":"ea","unitCost":220.00,"unitPrice":399.00,"leadTimeDays":8,"reorderPoint":25,"reorderQty":40},
    {"sku":"FOIL-3","name":"Hydrofoil Assembly v3","uom":"ea","unitCost":380.00,"unitPrice":599.00,"leadTimeDays":9,"reorderPoint":25,"reorderQty":40},
    {"sku":"CHR-1","name":"Fast Charger 2kW","uom":"ea","unitCost":95.00,"unitPrice":179.00,"leadTimeDays":5,"reorderPoint":30,"reorderQty":60},
])

inventory = pd.DataFrame([
    {"sku":"BF-100","warehouse":"SJ-01","onHand":14,"reserved":0,"inTransit":0},
    {"sku":"BAT-9","warehouse":"SJ-01","onHand":28,"reserved":0,"inTransit":10},
    {"sku":"CTRL-2","warehouse":"SJ-01","onHand":19,"reserved":0,"inTransit":0},
    {"sku":"FOIL-3","warehouse":"SJ-01","onHand":26,"reserved":0,"inTransit":0},
    {"sku":"CHR-1","warehouse":"SJ-01","onHand":32,"reserved":0,"inTransit":0},
])

# Simple BOM (one level for speed)
bom = pd.DataFrame([
    {"parent":"BF-100","component":"BAT-9","qtyPer":1},
    {"parent":"BF-100","component":"CTRL-2","qtyPer":1},
    {"parent":"BF-100","component":"FOIL-3","qtyPer":1},
    {"parent":"BF-100","component":"CHR-1","qtyPer":1},
])

# Customers
customers = pd.DataFrame([
    {"customerId":"C100","name":"Sunny Surf Co.","terms":"NET30"},
    {"customerId":"C200","name":"Wave Riders LLC","terms":"NET30"},
    {"customerId":"C300","name":"Direct-to-Consumer","terms":"CC"},
])

# Sample orders for the next two weeks (some will exceed stock)
orderLines = pd.DataFrame([
    {"orderId":"SO-001","orderDate":today + pd.Timedelta(days=1),"customerId":"C100","sku":"BF-100","qty":8},
    {"orderId":"SO-002","orderDate":today + pd.Timedelta(days=2),"customerId":"C200","sku":"BF-100","qty":7},
    {"orderId":"SO-003","orderDate":today + pd.Timedelta(days=2),"customerId":"C300","sku":"BF-100","qty":6},
    {"orderId":"SO-004","orderDate":today + pd.Timedelta(days=3),"customerId":"C100","sku":"BF-100","qty":9},
])

# Vendors (with messy/duplicate data on purpose)
vendors = pd.DataFrame([
    {"vendorId":"V01","name":"Voltix Power","phone":"(408) 555-0100","email":"orders@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V01X","name":"Voltix  Power ","phone":"408-555-0100","email":"sales@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V02","name":"FoilWorks","phone":"+1 408.555.0200","email":"contact@foilworks.io","defaultSku":"FOIL-3"},
    {"vendorId":"V03","name":"Controllytics","phone":"4085550300","email":"hello@controllytics.ai","defaultSku":"CTRL-2"},
    {"vendorId":"V04","name":"ChargeFast","phone":"(408)-555-0400","email":"orders@chargefast.co","defaultSku":"CHR-1"},
])

leadTimes = dict(zip(products["sku"], products["leadTimeDays"]))

# Simple GL (very lightweight for the exercise)
glAccounts = pd.DataFrame([
    {"account":"1200","name":"Accounts Receivable"},
    {"account":"1400","name":"Inventory"},
    {"account":"2000","name":"Accounts Payable"},
    {"account":"4000","name":"Revenue"},
    {"account":"5000","name":"COGS"},
])

CHECKS = {}   # each CHECK cell records PASS here; the validation cell reads it

print("Dataset loaded for", today.date(), "| look at the tables: products, inventory, orderLines, customers, vendors, bom")

Dataset loaded for 2026-09-23 | look at the tables: products, inventory, orderLines, customers, vendors, bom


## Part 1) Map the two big flows (about 12 minutes, written)

BlueFin sells directly to consumers and to specialty surf dealers. The team is worried about stockouts and
component shortages. Before you write code, map the two flows the code will touch.

- **Order-to-Cash (O2C):** sales order, allocation, pick/pack/ship, customer invoice, cash receipt, GL posting.
- **Procure-to-Pay (P2P):** purchase requisition, purchase order, goods receipt, vendor (AP) invoice, payment, GL posting.

Fill in the dictionary below. Answer each field in one or two complete sentences, using BlueFin's own data where you can.

- `*_modules`: name the ERP modules the flow uses (for example Sales and Distribution, Inventory, Procurement, Finance/AR, Finance/AP, General Ledger).
- `*_master_data`: name the master data the flow reads, and the BlueFin table that holds it (for example `customers`).
- `*_risk`: name one thing that can go wrong in the flow, specific to BlueFin.
- `*_control`: name one control that prevents or catches that risk, and say who or what performs it.
- `*_map`: write the flow as one line of steps joined by `->`, and in brackets name the document or record each step creates. Use at least four arrows.
  The format looks like this: `Step one [document] -> Step two [document] -> ...`

In [4]:
P1 = {
    "o2c_modules":     "REPLACE",
    "o2c_master_data": "REPLACE",
    "o2c_risk":        "REPLACE",
    "o2c_control":     "REPLACE",
    "o2c_map":         "REPLACE",

    "p2p_modules":     "REPLACE",
    "p2p_master_data": "REPLACE",
    "p2p_risk":        "REPLACE",
    "p2p_control":     "REPLACE",
    "p2p_map":         "REPLACE",
}

## Part 2) Order-to-Cash: allocate stock, then invoice (about 12 minutes)

Implement `allocateInventory` and `createInvoices` for the four BF-100 sales orders in `orderLines`.

**Rules for `allocateInventory`**
- Serve orders first come, first served: sort by `orderDate`, then by `orderId`. The scaffold does this for you.
- Start from the `onHand` quantity in `inventory` for each SKU. Keep the running amount in a dict.
- Give each order `min(qty, still available)`. Put the rest in `backorderedQty`. Subtract what you gave from the dict.
- Do not change the `inventory` table itself. Re-running a cell must not use up the stock twice.

**Rules for `createInvoices`**
- Create one invoice row for each line with `allocatedQty > 0`, in allocation order, numbered `INV-001`, `INV-002`, and so on.
- Look up `unitPrice` in `products` (use `getProductRecord`) and `terms` in `customers`.
- Set `amount = allocatedQty * unitPrice`. Set `invoiceDate` to the order's `orderDate` (assume the order ships that day).

Run the cell, then run the CHECK cell below it.

In [5]:
# Helper: get product price and cost quickly
def getProductRecord(sku: str) -> pd.Series:
    rec = products.loc[products["sku"] == sku]
    return rec.iloc[0] if not rec.empty else None

def allocateInventory(orderLinesDf: pd.DataFrame, inventoryDf: pd.DataFrame) -> pd.DataFrame:
    """
    Allocate on-hand inventory to orders by orderDate (FIFO).
    Returns a DataFrame with columns:
      orderId, orderDate, customerId, sku, qty, allocatedQty, backorderedQty
    """
    # TODO: Implement allocation logic
    # Steps:
    # 1) Sort orders by orderDate, then orderId (done below)
    # 2) Track available on-hand by sku in a dict (single warehouse for simplicity)
    # 3) For each order line, allocate up to available; compute backordered remainder
    # 4) Update the available bucket
    # 5) Return the DataFrame with allocated/backordered columns filled
    sortedLines = orderLinesDf.sort_values(["orderDate", "orderId"]).copy()
    sortedLines["allocatedQty"] = 0
    sortedLines["backorderedQty"] = sortedLines["qty"]  # default until allocated

    # >>> YOUR CODE HERE <<<

    return sortedLines

def createInvoices(allocationsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Create invoices for lines with allocatedQty > 0.
    Columns: invoiceId, orderId, customerId, sku, allocatedQty, unitPrice, amount, invoiceDate, terms
    """
    # TODO:
    # 1) Filter allocated lines
    # 2) Look up price (products) and customer terms (customers)
    # 3) Compute amount = allocatedQty * unitPrice; invoiceDate = orderDate
    # 4) Generate invoiceId like INV-001, INV-002, ...
    invRows = []

    # >>> YOUR CODE HERE <<<

    invoicesDf = pd.DataFrame(invRows, columns=["invoiceId","orderId","customerId","sku","allocatedQty","unitPrice","amount","invoiceDate","terms"])
    return invoicesDf

allocations = allocateInventory(orderLines, inventory)
invoices = createInvoices(allocations)
print(allocations[["orderId", "orderDate", "customerId", "qty", "allocatedQty", "backorderedQty"]].to_string(index=False))
print()
print(invoices.to_string(index=False) if len(invoices) else "(no invoices yet)")

orderId  orderDate customerId  qty  allocatedQty  backorderedQty
 SO-001 2026-09-24       C100    8             0               8
 SO-002 2026-09-25       C200    7             0               7
 SO-003 2026-09-25       C300    6             0               6
 SO-004 2026-09-26       C100    9             0               9

(no invoices yet)


In [6]:
# CHECK for Part 2. Run it after the Part 2 cell. Do not edit it.
# The expected values come from the instructor's reference solution run on the setup data.
EXPECTED_P2_ALLOC = [
    ('SO-001', 8, 8, 0),
    ('SO-002', 7, 6, 1),
    ('SO-003', 6, 0, 6),
    ('SO-004', 9, 0, 9),
]   # (orderId, qty, allocatedQty, backorderedQty) in FIFO order
EXPECTED_P2_INV = [
    ('INV-001', 'SO-001', 'C100', 8, 2999.0, 23992.0, '2026-09-24', 'NET30'),
    ('INV-002', 'SO-002', 'C200', 6, 2999.0, 17994.0, '2026-09-25', 'NET30'),
]       # (invoiceId, orderId, customerId, allocatedQty, unitPrice, amount, invoiceDate, terms)

def _check_part2():
    problems = []
    start = int(inventory.loc[inventory["sku"] == "BF-100", "onHand"].sum())
    before = inventory.copy()
    alloc = allocateInventory(orderLines, inventory)
    if not inventory.equals(before):
        problems.append("allocateInventory changed the inventory table. Copy the on-hand numbers into a dict "
                        "and subtract from the dict, so running the cell twice does not use the stock up twice.")
    missing = {"orderId", "qty", "allocatedQty", "backorderedQty"} - set(alloc.columns)
    if missing:
        return problems + [f"The allocation result is missing the column(s) {sorted(missing)}. Return the "
                           f"sortedLines DataFrame from the scaffold, with those columns filled in."]
    got = {r["orderId"]: (int(r["allocatedQty"]), int(r["backorderedQty"])) for _, r in alloc.iterrows()}
    taken = 0
    for oid, qty, a, b in EXPECTED_P2_ALLOC:
        left = start - taken
        if oid not in got:
            problems.append(f"{oid} is missing from the allocation result. Return one row for every order line.")
        elif got[oid] != (a, b):
            ga, gb = got[oid]
            state = ("should be filled in full" if a == qty else
                     ("should get no stock" if a == 0 else "should be partly filled"))
            if taken == 0:
                why = f"It is first by orderDate, so it can take up to all {start} on hand."
            else:
                why = (f"FIFO by orderDate means the earlier orders took {taken} of the {start} on hand, "
                       f"leaving {left} for {oid}.")
            problems.append(f"{oid} {state}: {a} of {qty} allocated, {b} backordered. "
                            f"Your code gives {ga} allocated and {gb} backordered. {why}")
            if ga + gb != qty:
                problems.append(f"For {oid}, allocatedQty + backorderedQty is {ga + gb}, but the order is for "
                                f"{qty}. Every unit ordered is either allocated or backordered.")
        taken += a
    if problems:
        problems.append("createInvoices was not checked yet, because invoices are built from the allocation. Fix the allocation first.")
        return problems

    inv = createInvoices(alloc)
    if len(inv) != len(EXPECTED_P2_INV):
        none = [oid for oid, qty, a, b in EXPECTED_P2_ALLOC if a == 0]
        return [f"There should be {len(EXPECTED_P2_INV)} invoices, one per order line that got stock; your code "
                f"made {len(inv)}. Orders {', '.join(none)} got 0 units, so they get no invoice."]
    for (iid, oid, cid, q, price, amount, idate, terms), (_, row) in zip(EXPECTED_P2_INV, inv.iterrows()):
        if str(row.get("invoiceId")) != iid or str(row.get("orderId")) != oid:
            problems.append(f"Invoice {iid} should be for {oid}, in allocation order. Yours is "
                            f"{row.get('invoiceId')} for {row.get('orderId')}. Number invoices INV-001, INV-002, and so on.")
            continue
        try:
            got_amount = float(row.get("amount"))
        except (TypeError, ValueError):
            got_amount = None
        if got_amount is None or abs(got_amount - amount) > 0.005:
            problems.append(f"{iid} amount should be {q} x {price:.2f} = {amount:.2f}; yours is {row.get('amount')}. "
                            f"Use allocatedQty, not the ordered qty, and the unitPrice from products.")
        if str(row.get("terms")) != terms:
            problems.append(f"{iid} terms should be {terms}, from the customers table for {cid}; yours is {row.get('terms')}.")
        try:
            same_day = pd.Timestamp(row.get("invoiceDate")).normalize() == pd.Timestamp(idate)
        except Exception:
            same_day = False
        if not same_day:
            problems.append(f"{iid} invoiceDate should be the order date {idate}; yours is {row.get('invoiceDate')}.")
    return problems

try:
    _problems = _check_part2()
except NameError as e:
    _problems = [f"{e}. Run the Part 2 code cell first, then run this CHECK again."]
except Exception as e:
    _problems = [f"Your Part 2 code raised {type(e).__name__}: {e}. Read the last line of the error, "
                 f"fix that line, run the Part 2 cell, then run this CHECK again."]
CHECKS["Part 2"] = not _problems
if _problems:
    for p in _problems:
        print("NOT YET:", p)
else:
    _bo = sum(b for _, _, _, b in EXPECTED_P2_ALLOC)
    _total = sum(x[5] for x in EXPECTED_P2_INV)
    print(f"PASS: allocation matches FIFO ({_bo} units backordered) and {len(EXPECTED_P2_INV)} invoices total {_total:,.2f}.")

NOT YET: SO-001 should be filled in full: 8 of 8 allocated, 0 backordered. Your code gives 0 allocated and 8 backordered. It is first by orderDate, so it can take up to all 14 on hand.
NOT YET: SO-002 should be partly filled: 6 of 7 allocated, 1 backordered. Your code gives 0 allocated and 7 backordered. FIFO by orderDate means the earlier orders took 8 of the 14 on hand, leaving 6 for SO-002.
NOT YET: createInvoices was not checked yet, because invoices are built from the allocation. Fix the allocation first.


**Part 2 written answer.** In one or two sentences, name the orders that are backordered and by how many
units in total, and name the ERP step that would fill the backorder. If your code would not run, start the
answer with `BROKE:` and paste the error message.

In [7]:
P2 = {
    "backorder_next_step": "REPLACE",
}

## Part 3) Clean the vendor master (about 12 minutes)

The `vendors` table has a duplicate, extra spaces, and five different phone formats. Implement `cleanVendorMaster`.

**Rules**
- Clean each name: remove leading and trailing spaces and collapse repeated spaces (`" ".join(name.split())`).
  Keep the capitalization the vendor uses: `FoilWorks` stays `FoilWorks`. Use a lowercase copy only inside your dedupe key.
- Standardize every phone with the `normalizePhone` helper, to the form `+14085550100`.
- De-duplicate so the two Voltix Power records become one row. For the merged row, keep the `vendorId` of the
  first record in the table, and keep the longest email and the longest phone.
- Return the columns `vendorId, name, phone, email, defaultSku`, one row per real vendor.
- Do not change the `vendors` table itself.

Run the cell, then run the CHECK cell below it.

In [8]:
def normalizePhone(phone: str) -> str:
    # Given: keeps the digits and adds the country code
    digits = "".join(ch for ch in str(phone) if ch.isdigit())
    if digits.startswith("1") and len(digits) == 11:
        return "+" + digits
    if len(digits) == 10:
        return "+1" + digits
    return "+" + digits if digits else ""

def cleanVendorMaster(vendorsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Return a cleaned and de-duplicated vendor master.
    Hints:
      - " ".join(name.split()) cleans the spaces; keep the vendor's capitalization
      - .apply(normalizePhone) or .map(normalizePhone)
      - groupby on a dedupe key you design (for example a lowercase copy of the clean name)
      - in each group keep the first vendorId and the longest email and phone
    """
    cleaned = vendorsDf.copy()

    # TODO: implement cleaning + dedupe
    # >>> YOUR CODE HERE <<<

    return cleaned

vendors_clean = cleanVendorMaster(vendors)
print(vendors_clean.to_string(index=False))

vendorId           name           phone                  email defaultSku
     V01   Voltix Power  (408) 555-0100      orders@voltix.com      BAT-9
    V01X Voltix  Power     408-555-0100       sales@voltix.com      BAT-9
     V02      FoilWorks +1 408.555.0200   contact@foilworks.io     FOIL-3
     V03  Controllytics      4085550300 hello@controllytics.ai     CTRL-2
     V04     ChargeFast  (408)-555-0400   orders@chargefast.co      CHR-1


In [9]:
# CHECK for Part 3. Run it after the Part 3 cell. Do not edit it.
# The expected rows come from the instructor's reference solution run on the setup data.
EXPECTED_P3 = [
    ('V01', 'Voltix Power', '+14085550100', 'orders@voltix.com', 'BAT-9'),
    ('V02', 'FoilWorks', '+14085550200', 'contact@foilworks.io', 'FOIL-3'),
    ('V03', 'Controllytics', '+14085550300', 'hello@controllytics.ai', 'CTRL-2'),
    ('V04', 'ChargeFast', '+14085550400', 'orders@chargefast.co', 'CHR-1'),
]   # (vendorId, name, phone, email, defaultSku)

def _check_part3():
    problems = []
    before = vendors.copy()
    got = cleanVendorMaster(vendors)
    if not vendors.equals(before):
        problems.append("cleanVendorMaster changed the vendors table. Work on the copy the scaffold makes (cleaned).")
    cols = ["vendorId", "name", "phone", "email", "defaultSku"]
    missing = set(cols) - set(got.columns)
    if missing:
        return problems + [f"The result is missing the column(s) {sorted(missing)}. Return vendorId, name, phone, email and defaultSku."]
    n = len(EXPECTED_P3)
    if len(got) > n:
        dup = [e[0] for e in EXPECTED_P3 if "Voltix" in e[1]]
        return problems + [f"You have {len(got)} vendor rows; there should be {n}. 'Voltix Power' and 'Voltix  Power ' "
                           f"are one company with extra spaces in the name, so they must merge into one row "
                           f"({dup[0] if dup else 'V01'}). Clean the spaces before you group, and group on the clean key."]
    if len(got) < n:
        return problems + [f"You have {len(got)} vendor rows; there should be {n}. Only the two Voltix records are "
                           f"duplicates. Your dedupe key merged different vendors, so make it more specific."]
    rows = {str(r["vendorId"]): r for _, r in got.iterrows()}
    for vid, name, phone, email, sku in EXPECTED_P3:
        if vid not in rows:
            other = ", ".join(sorted(rows))
            problems.append(f"There is no row with vendorId {vid}; your ids are {other}. The merged row keeps the "
                            f"vendorId of the first record in the table.")
            continue
        r = rows[vid]
        gname = str(r["name"])
        if gname != name:
            if gname.strip().lower() == name.lower() and " ".join(gname.split()) == gname:
                problems.append(f"{vid} name came back as '{gname}', but the vendor spells it '{name}'. str.title() "
                                f"and str.lower() rewrite a company's name. Keep the name as given and use a "
                                f"lowercase copy only in the dedupe key.")
            else:
                problems.append(f"{vid} name should be '{name}'; yours is '{gname}'. Remove the outer spaces and "
                                f"collapse repeated spaces with ' '.join(name.split()).")
        if str(r["phone"]) != phone:
            problems.append(f"{vid} phone should be {phone}; yours is '{r['phone']}'. Apply normalizePhone to "
                            f"every phone.")
        if str(r["email"]).strip().lower() != email:
            problems.append(f"{vid} email should be {email}; yours is '{r['email']}'. When records merge, keep "
                            f"the longest email.")
        if str(r["defaultSku"]) != sku:
            problems.append(f"{vid} defaultSku should be {sku}; yours is '{r['defaultSku']}'.")
    return problems

try:
    _problems = _check_part3()
except NameError as e:
    _problems = [f"{e}. Run the Part 3 code cell first, then run this CHECK again."]
except Exception as e:
    _problems = [f"Your Part 3 code raised {type(e).__name__}: {e}. Read the last line of the error, "
                 f"fix that line, run the Part 3 cell, then run this CHECK again."]
CHECKS["Part 3"] = not _problems
if _problems:
    for p in _problems:
        print("NOT YET:", p)
else:
    print(f"PASS: {len(vendors)} vendor records cleaned to {len(EXPECTED_P3)}, with every phone in +1 form.")

NOT YET: You have 5 vendor rows; there should be 4. 'Voltix Power' and 'Voltix  Power ' are one company with extra spaces in the name, so they must merge into one row (V01). Clean the spaces before you group, and group on the clean key.


**Part 3 written answer.** Name the dedupe key your code groups on. In one or two sentences, say why that
key finds the two Voltix records and does not merge two different vendors, and why `vendorId` or `email`
would not work as the key. If your code would not run, start the answer with `BROKE:` and paste the error message.

In [10]:
P3 = {
    "dedupe_key_and_why": "REPLACE",
}

## Part 4) Roles and segregation of duties (about 12 minutes)

BlueFin uses these roles. Segregation of duties (SoD) means no single person holds two actions that together allow fraud.

- **SalesRep:** create orders, view inventory.
- **Warehouse:** allocate, ship goods, view inventory.
- **Buyer:** create POs, receive goods, view inventory.
- **APClerk:** enter vendor invoices, schedule payments.
- **Controller:** approve payments, post period close, view inventory.

**Rules**
- `canPerform(role, action)` returns `True` if the role includes the action. It returns `False` for a role
  that is not in `roles`, and it does not raise an error.
- `hasConflict(userRoles)` takes a set of role names held by one person. It combines all their actions, and
  returns the `reason` of every rule in `sodConflicts` whose actions are all covered, in the order of `sodConflicts`.
  It returns an empty list when nothing matches.

Run the cell, then run the CHECK cell below it.

In [11]:
roles = {
    "SalesRep": {"createSO","viewInv"},
    "Warehouse": {"allocate","ship","viewInv"},
    "Buyer": {"createPO","receive","viewInv"},
    "APClerk": {"enterAP","schedulePay"},
    "Controller": {"approvePay","postClose","viewInv"},
}

sodConflicts = [
    {"actions":{"enterAP","approvePay"}, "reason":"Same person could create and approve vendor payments."},
    {"actions":{"createSO","ship"}, "reason":"Same person could create and ship a fraudulent order."},
]

def canPerform(role: str, action: str) -> bool:
    """Return True if role includes the action."""
    # TODO: implement check
    # >>> YOUR CODE HERE <<<
    return False

def hasConflict(userRoles: set) -> list:
    """
    Given a set of roles assigned to a user, return a list of conflict reasons matched.
    A conflict matches if the union of actions across those roles covers all actions in a conflict rule.
    """
    # TODO: implement SoD conflict detection
    # >>> YOUR CODE HERE <<<
    return []

print(canPerform("SalesRep", "viewInv"), canPerform("SalesRep", "ship"))
print(hasConflict({"APClerk", "Controller"}))
print(hasConflict({"Buyer", "APClerk"}))

False False
[]
[]


In [12]:
# CHECK for Part 4. Run it after the Part 4 cell. Do not edit it.
# The expected answers come from the instructor's reference solution run on the roles above.
EXPECTED_P4_CAN = [
    ('SalesRep', 'viewInv', True),
    ('SalesRep', 'ship', False),
    ('Warehouse', 'ship', True),
    ('APClerk', 'approvePay', False),
    ('Auditor', 'viewInv', False),
]             # (role, action, expected)
EXPECTED_P4_CONFLICT = [
    (['APClerk', 'Controller'], ['Same person could create and approve vendor payments.']),
    (['SalesRep', 'Warehouse'], ['Same person could create and ship a fraudulent order.']),
    (['APClerk'], []),
    (['APClerk', 'Buyer'], []),
    (['APClerk', 'Controller', 'SalesRep', 'Warehouse'], ['Same person could create and approve vendor payments.', 'Same person could create and ship a fraudulent order.']),
]   # (roles held by one person, expected reasons)

def _check_part4():
    problems = []
    for role, action, want in EXPECTED_P4_CAN:
        try:
            got = canPerform(role, action)
        except KeyError:
            problems.append(f"canPerform('{role}', '{action}') raised KeyError. '{role}' is not a role in the table; "
                            f"return False for it. roles.get(role, set()) gives an empty set for an unknown role.")
            continue
        if got is not want and got != want:
            if role not in roles:
                why = f"'{role}' is not in roles, so it can do nothing."
            elif want:
                why = f"{role} has {sorted(roles[role])}, which includes '{action}'."
            else:
                why = f"{role} has {sorted(roles[role])}, which does not include '{action}'."
            problems.append(f"canPerform('{role}', '{action}') should be {want}; yours is {got}. {why}")
    for held, want in EXPECTED_P4_CONFLICT:
        got = hasConflict(set(held))
        if not isinstance(got, list):
            problems.append(f"hasConflict({set(held)}) should return a list of reasons; yours returns a {type(got).__name__}.")
            continue
        if got != want:
            acts = sorted(set().union(*(roles.get(r, set()) for r in held)))
            if not want:
                why = (f"Together these roles hold {acts}. No rule has all of its actions in that list, so nothing "
                       f"matches. A rule matches only when every action in it is covered, not just one.")
            else:
                why = (f"Together these roles hold {acts}, which covers every action in {len(want)} rule(s). "
                       f"Combine the actions of all the roles first, then test each rule with <= (subset).")
            problems.append(f"hasConflict({sorted(held)}) should return {want}; yours returns {got}. {why}")
    return problems

try:
    _problems = _check_part4()
except NameError as e:
    _problems = [f"{e}. Run the Part 4 code cell first, then run this CHECK again."]
except Exception as e:
    _problems = [f"Your Part 4 code raised {type(e).__name__}: {e}. Read the last line of the error, "
                 f"fix that line, run the Part 4 cell, then run this CHECK again."]
CHECKS["Part 4"] = not _problems
if _problems:
    for p in _problems:
        print("NOT YET:", p)
else:
    print(f"PASS: {len(EXPECTED_P4_CAN)} access checks and {len(EXPECTED_P4_CONFLICT)} SoD cases are right.")

NOT YET: canPerform('SalesRep', 'viewInv') should be True; yours is False. SalesRep has ['createSO', 'viewInv'], which includes 'viewInv'.
NOT YET: canPerform('Warehouse', 'ship') should be True; yours is False. Warehouse has ['allocate', 'ship', 'viewInv'], which includes 'ship'.
NOT YET: hasConflict(['APClerk', 'Controller']) should return ['Same person could create and approve vendor payments.']; yours returns []. Together these roles hold ['approvePay', 'enterAP', 'postClose', 'schedulePay', 'viewInv'], which covers every action in 1 rule(s). Combine the actions of all the roles first, then test each rule with <= (subset).
NOT YET: hasConflict(['SalesRep', 'Warehouse']) should return ['Same person could create and ship a fraudulent order.']; yours returns []. Together these roles hold ['allocate', 'createSO', 'ship', 'viewInv'], which covers every action in 1 rule(s). Combine the actions of all the roles first, then test each rule with <= (subset).
NOT YET: hasConflict(['APClerk', 

**Part 4 written answer.** `hasConflict({"Buyer", "APClerk"})` returns no conflict. In one or two
sentences, say what one person holding both roles could do, and write the SoD rule (the two actions) that
would catch it. If your code would not run, start the answer with `BROKE:` and paste the error message.

In [13]:
P4 = {
    "missing_rule": "REPLACE",
}

<details>
<summary><strong>Hints (expand if your team is stuck)</strong></summary>

**Part 1 (process map):**
- O2C modules: Sales and Distribution, Inventory, Finance (Accounts Receivable), General Ledger. P2P modules: Procurement, Inventory, Finance (Accounts Payable), General Ledger.
- Master data is the slow-changing reference data a transaction reads: customers, products, price lists and payment terms for O2C; vendors, products and GL accounts for P2P.
- A good control names who or what performs it: a credit check before release, a three-way match of PO, receipt and invoice, an approval that a second person gives.

**Part 2 (allocation and invoices):**
- Track stock in a dict: `available = inventoryDf.groupby("sku")["onHand"].sum().to_dict()`.
- For each line: `take = min(qty, available[sku])`, backorder `qty - take`, then `available[sku] -= take`.
- Write into the DataFrame with `sortedLines.at[idx, "allocatedQty"] = take` inside `for idx, line in sortedLines.iterrows():`.
- Terms: `terms = dict(zip(customers["customerId"], customers["terms"]))`. Invoice ids: `f"INV-{n:03d}"`.

**Part 3 (vendor master):**
- Clean a name with `" ".join(name.split())`. Do not use `.str.title()`: it turns FoilWorks into Foilworks.
- Build a key column, for example `cleaned["key"] = cleaned["name"].str.casefold()`, then `for key, grp in cleaned.groupby("key", sort=False):`.
- In each group keep `grp.iloc[0]["vendorId"]` and `max(grp["email"], key=len)`.

**Part 4 (roles and SoD):**
- `canPerform`: `return action in roles.get(role, set())`.
- `hasConflict`: combine the actions with `set().union(*(roles.get(r, set()) for r in userRoles))`, then keep each rule whose `rule["actions"] <= actions`.

**Optional A and B:**
- Available = onHand + inTransit - reserved. Match a vendor with `vendors.loc[vendors["defaultSku"] == sku].head(1)`.
- MRP shortage = `max(0, grossReq - available)`; planned release = shortage.

</details>

## Validation: run this before you submit

This cell checks that the notebook is complete. It does not grade your answers. Fix everything it lists,
then choose **Runtime > Restart and run all** and run it again.

In [14]:
# Validation (plain Python). Do not edit.
problems = []

def _filled(v):
    return isinstance(v, str) and v.strip() != "" and "REPLACE" not in v

VAGUE = {
    "it depends": "\"it depends\" is not an answer. Say what it depends on, and what BlueFin should do.",
    "depends": "\"depends\" is not an answer. Say what it depends on, and what BlueFin should do.",
    "n/a": "\"n/a\" says the question does not apply. Every question here applies to BlueFin. Answer it, or write BROKE: and what stopped you.",
    "na": "\"na\" says the question does not apply. Every question here applies to BlueFin. Answer it, or write BROKE: and what stopped you.",
    "none": "\"none\" is not an answer. Name the specific item the question asks for.",
    "tbd": "\"tbd\" means the answer is not written yet. Write it before you submit.",
    "todo": "\"todo\" means the answer is not written yet. Write it before you submit.",
    "idk": "\"idk\" is not an answer. Discuss it as a team and write your best answer, or write BROKE: and what stopped you.",
    "?": "\"?\" is not an answer. Write your team's best answer in a complete sentence.",
    "see above": "\"see above\" is not an answer. Write the answer here, so it can be read on its own.",
    "same": "\"same\" is not an answer. Write the answer here, so it can be read on its own.",
    "yes": "\"yes\" is not an answer. Write a complete sentence that says what and why.",
    "no": "\"no\" is not an answer. Write a complete sentence that says what and why.",
}

def _answer(v, where, minwords=6):
    if not _filled(v):
        problems.append(f"{where}: still says REPLACE. Write your answer in its place.")
        return
    text = v.strip()
    if text.startswith("BROKE:"):
        if len(text.split()) < 3:
            problems.append(f"{where}: after BROKE:, paste the error message and say where it happened.")
        return
    key = text.lower().rstrip(".!").strip()
    if key in VAGUE:
        problems.append(f"{where}: {VAGUE[key]}")
    elif len(text.split()) < minwords:
        problems.append(f"{where}: \"{text}\" is {len(text.split())} word(s). Write at least {minwords} words, as a complete sentence.")

# team
try:
    GROUP
except NameError:
    GROUP = []
    problems.append("GROUP is not defined. Run the team cell at the top of the notebook.")
names = [g.strip() for g in GROUP if isinstance(g, str)]
if not 2 <= len(names) <= 4:
    problems.append(f"GROUP has {len(names)} name(s). Teams have 2 to 4 members; add or delete lines so the list matches your team.")
for i, g in enumerate(names, 1):
    if not _filled(g):
        problems.append(f"GROUP entry {i}: still says REPLACE. Write the member's name as it appears in Canvas.")
_seen = [g.lower() for g in names if _filled(g)]
for g in sorted(set(x for x in _seen if _seen.count(x) > 1)):
    problems.append(f"GROUP lists '{g}' more than once. Write each member once.")
if not _filled(globals().get("TEAM_NAME", "")):
    problems.append("TEAM_NAME: still says REPLACE. Write your project team's name.")

# Part 1
_P1 = globals().get("P1", {})
for flow in ("o2c", "p2p"):
    _answer(_P1.get(f"{flow}_modules", "REPLACE"), f"Part 1 {flow}_modules", minwords=3)
    _answer(_P1.get(f"{flow}_master_data", "REPLACE"), f"Part 1 {flow}_master_data", minwords=3)
    _answer(_P1.get(f"{flow}_risk", "REPLACE"), f"Part 1 {flow}_risk")
    _answer(_P1.get(f"{flow}_control", "REPLACE"), f"Part 1 {flow}_control")
    m = _P1.get(f"{flow}_map", "REPLACE")
    if not _filled(m):
        problems.append(f"Part 1 {flow}_map: still says REPLACE. Write the flow as one line of steps joined by ->.")
    elif not m.strip().startswith("BROKE:") and m.count("->") < 4:
        problems.append(f"Part 1 {flow}_map: has {m.count('->')} arrow(s). Write at least four -> arrows, one between each pair of steps.")

# Parts 2 to 4: the written answer, and the CHECK
_CHECKS = globals().get("CHECKS", {})
for part, d, key in (("Part 2", "P2", "backorder_next_step"),
                     ("Part 3", "P3", "dedupe_key_and_why"),
                     ("Part 4", "P4", "missing_rule")):
    v = globals().get(d, {}).get(key, "REPLACE")
    _answer(v, f"{part} {key}")
    broke = isinstance(v, str) and v.strip().startswith("BROKE:")
    if part not in _CHECKS:
        problems.append(f"{part}: the CHECK cell has not run. Run it, or choose Runtime > Restart and run all.")
    elif not _CHECKS[part] and not broke:
        problems.append(f"{part}: the CHECK did not print PASS. Fix the code until it does, or start {d}['{key}'] "
                        f"with BROKE: and paste the error message.")

if problems:
    print("NOT READY")
    for p in problems:
        print(" -", p)
else:
    print("READY. To submit:")
    print(" 1. Choose Runtime > Restart and run all, and confirm that every cell has output.")
    print(" 2. Choose File > Download > Download .ipynb.")
    print(" 3. One team member uploads that file to the Canvas assignment In-Class: ERP Lab by 11:59 PM tonight.")

NOT READY
 - Part 1 o2c_modules: still says REPLACE. Write your answer in its place.
 - Part 1 o2c_master_data: still says REPLACE. Write your answer in its place.
 - Part 1 o2c_risk: still says REPLACE. Write your answer in its place.
 - Part 1 o2c_control: still says REPLACE. Write your answer in its place.
 - Part 1 o2c_map: still says REPLACE. Write the flow as one line of steps joined by ->.
 - Part 1 p2p_modules: still says REPLACE. Write your answer in its place.
 - Part 1 p2p_master_data: still says REPLACE. Write your answer in its place.
 - Part 1 p2p_risk: still says REPLACE. Write your answer in its place.
 - Part 1 p2p_control: still says REPLACE. Write your answer in its place.
 - Part 1 p2p_map: still says REPLACE. Write the flow as one line of steps joined by ->.
 - Part 2 backorder_next_step: still says REPLACE. Write your answer in its place.
 - Part 2: the CHECK did not print PASS. Fix the code until it does, or start P2['backorder_next_step'] with BROKE: and paste t

## Summary: this is what the grader reads first

In [15]:
print("Team:", globals().get("TEAM_NAME"), "|", ", ".join(str(g) for g in globals().get("GROUP", [])))
print()
for part in ("Part 2", "Part 3", "Part 4"):
    state = globals().get("CHECKS", {}).get(part)
    print(f"{part} CHECK:", "PASS" if state else ("NOT YET" if state is False else "not run"))
for name in ("P1", "P2", "P3", "P4"):
    print()
    print(name)
    for key, val in globals().get(name, {}).items():
        print(f"  {key}: {val}")

Team: HIKE THE MODEL | AMEYA MATHEW, Olivia Hunter, Salih Eren Yüzbaşıoğlu

Part 2 CHECK: NOT YET
Part 3 CHECK: NOT YET
Part 4 CHECK: NOT YET

P1
  o2c_modules: REPLACE
  o2c_master_data: REPLACE
  o2c_risk: REPLACE
  o2c_control: REPLACE
  o2c_map: REPLACE
  p2p_modules: REPLACE
  p2p_master_data: REPLACE
  p2p_risk: REPLACE
  p2p_control: REPLACE
  p2p_map: REPLACE

P2
  backorder_next_step: REPLACE

P3
  dedupe_key_and_why: REPLACE

P4
  missing_rule: REPLACE


---
### What to submit
One team member uploads the downloaded `.ipynb` to the Canvas assignment **In-Class: ERP Lab** by 11:59 PM tonight.
Before you download, the validation cell must print READY, and Runtime > Restart and run all must finish with output in every cell.

---
# Optional stretch (not graded)

Do these only if your team finishes Parts 1 to 4 and the validation cell prints READY. They are not graded
and the validator ignores them.

## Optional A) Procure-to-Pay: reorders and receiving

**Rules**
- For each SKU, compute `available = onHand + inTransit - reserved`.
- If `available < reorderPoint`, propose a PO for `reorderQty`, with `etaDate = today + leadTimeDays`.
  Guess the vendor from the first row of `vendors` whose `defaultSku` matches.
- Placing a PO puts its quantity in transit. That step is written for you in `receivePo`.
  Receiving then moves the quantity from `inTransit` to `onHand`. Write that step.

Then write two sentences in the text cell of your choice: which GL accounts the receipt, the vendor invoice,
and the payment would post to, using the accounts in `glAccounts`.

In [16]:
def suggestReorders(inventoryDf: pd.DataFrame, productsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame of suggested POs with columns:
      poId, sku, qty, etaDate, vendorGuess
    Vendor is guessed from `vendors.defaultSku` (simplified!)
    """
    # TODO:
    # 1) For each sku, compute available
    # 2) If below reorderPoint, propose reorderQty and ETA today + leadTimeDays
    # 3) Guess vendor by matching vendors.defaultSku == sku (first hit)
    poRows = []
    poCounter = 1

    # >>> YOUR CODE HERE <<<

    return pd.DataFrame(poRows, columns=["poId","sku","qty","etaDate","vendorGuess"])

def receivePo(inventoryDf: pd.DataFrame, poDf: pd.DataFrame) -> pd.DataFrame:
    """
    Places the POs (given), then receives them today.
    Returns an updated copy of inventoryDf.
    """
    inv = inventoryDf.copy()
    # Given: placing each PO puts its qty in transit.
    for _, po in poDf.iterrows():
        inv.loc[(inv["sku"] == po["sku"]) & (inv["warehouse"] == "SJ-01"), "inTransit"] += int(po["qty"])
    # TODO: receive each PO: move its qty from inTransit to onHand for the matching sku at SJ-01
    # >>> YOUR CODE HERE <<<
    return inv

# --- Quick test harness (uncomment after you implement) ---
# pos = suggestReorders(inventory, products)
# print(pos.to_string(index=False))
# inventory_after = receivePo(inventory, pos)
# print(inventory_after.to_string(index=False))

## Optional B) A one-level MRP for BF-100

A dealer asks for 30 BF-100 kits by next Friday. Explode the bill of materials and plan the components.

**Rules**
- Gross requirement for each component = BF-100 demand x `qtyPer` in `bom`.
- `available = onHand + inTransit - reserved`. Shortage = `max(0, grossReq - available)`. The planned release quantity is the shortage (no lot sizing).
- `releaseDate = needDate - leadTimeDays`.

When you run it, look at the release dates. A release date before today (September 23) means the order is
already late. Write one sentence on what BlueFin should do about it.

In [17]:
def planMrpForBf100(demandQty: int, needDate: pd.Timestamp) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      component, grossReq, available, shortage, plannedReleaseQty, releaseDate
    """
    rows = []
    for _, r in bom[bom["parent"]=="BF-100"].iterrows():
        comp = r["component"]
        qtyPer = r["qtyPer"]
        gross = demandQty * qtyPer

        # Compute available
        invRow = inventory[(inventory["sku"]==comp) & (inventory["warehouse"]=="SJ-01")]
        available = int(invRow["onHand"].iloc[0] + invRow["inTransit"].iloc[0] - invRow["reserved"].iloc[0])

        # TODO: compute shortage and planned release by comparing to gross
        # >>> YOUR CODE HERE <<<
        shortage = None
        plannedReleaseQty = None

        # compute releaseDate = needDate - leadTimeDays
        lt = leadTimes.get(comp, 0)
        releaseDate = (needDate - pd.Timedelta(days=int(lt))).normalize()

        rows.append({
            "component": comp,
            "grossReq": gross,
            "available": available,
            "shortage": shortage,
            "plannedReleaseQty": plannedReleaseQty,
            "releaseDate": releaseDate,
        })
    return pd.DataFrame(rows)

# --- Quick test harness (uncomment after you implement) ---
# next_friday = (today + pd.offsets.Week(weekday=4)).normalize()
# mrp = planMrpForBf100(demandQty=30, needDate=next_friday)
# print(mrp.to_string(index=False))

## Optional C) A fulfillment KPI plot

After Part 2 passes, uncomment the cell to plot allocated against backordered BF-100 units by order date.

In [18]:
# daily = allocations.groupby(allocations["orderDate"].dt.normalize())[["allocatedQty","backorderedQty"]].sum()
# daily.plot()
# plt.title("Daily BF-100: Allocated vs Backordered")
# plt.xlabel("Date")
# plt.ylabel("Units")
# plt.show()